In [ ]:
# validate data statements, starting with birth date.

import json
import pandas
import pathlib
import pydash
import requests
import tqdm

def value_extract(row, column):

    ''' Extract dictionary values. '''
    
    return pydash.get(row[column], 'value')

def sparql_query(query, service):

    ''' Send sparql request, and formulate results into a dataframe. '''

    response = requests.get(service, params={'format': 'json', 'query': query}, timeout=120)
    results = pydash.get(response.json(), 'results.bindings')
    df = pandas.DataFrame.from_dict(results)
    for column in df.columns:
        df[column] = df.apply(value_extract, column=column, axis=1)
    
    return df

# birthdates for synced creators from wikidata.

query = ''' 
    select ?acmi_creator ?wikidata_birthdate where { 
        ?wd wdt:P7003 ?acmi_creator .
        filter(regex(str(?acmi_creator), "creator"))
        ?wd wdt:P569 ?wikidata_birthdate .
    } '''

wikidata_birth = sparql_query(query, 'https://query.wikidata.org/sparql').drop_duplicates()
wikidata_birth['wikidata_birthdate'] = wikidata_birth['wikidata_birthdate'].str[:10]

print(len(wikidata_birth))
wikidata_birth.head()

In [ ]:
acmi_birth = pandas.read_csv(pathlib.Path.cwd().parents[0] / 'acmi-api' / 'app' / 'tsv' / 'creators.tsv', delimiter='\t')
acmi_birth['id'] = 'creators/'+acmi_birth['id'].astype(str)
acmi_birth = acmi_birth[['id', 'date_of_birth']].rename(columns={'id':'acmi_creator', 'date_of_birth':'acmi_birthdate'})
acmi_birth = acmi_birth.dropna()
acmi_birth = acmi_birth[acmi_birth.acmi_birthdate.apply(lambda x: len(str(x))==10)]

print(len(acmi_birth))
acmi_birth.head()

In [ ]:
compare = pandas.merge(acmi_birth, wikidata_birth, on='acmi_creator', how='inner')
compare = compare.loc[compare.acmi_birthdate != compare.wikidata_birthdate]
print(len(compare))
compare.head()